# MoViNet-A1 Stream Training — dataset_scoop (normal vs steal), dựa trên V5 — Aligned với official recipe

Sinh từ `movinet-stream-optimized-v3.ipynb` bởi `make_v5.py`.

V5 align với [official YAML movinet_a1_stream_k600_8x8.yaml](https://github.com/tensorflow/models/blob/master/official/projects/movinet/configs/yaml/movinet_a1_stream_k600_8x8.yaml):
- **50 frames × stride 5** (match pretrain 10s), zero-pad clip ngắn
- **CCE + label_smoothing 0.1** (bỏ focal loss — over-engineer cho imbalance mild)
- **stochastic_depth_drop_rate=0.2** (official regularizer, V4 thiếu)
- **AdamW weight_decay=3e-5** (match official L2)
- **EMA decay 0.99** (Polyak paper)
- **2 stages** thay 3: head-only 4ep + full fine-tune 18ep
- Random resized crop (official-style, bỏ cutout + temporal reverse)
- TTA 10-crop làm report metric (1-crop cho model selection)

**Chạy trên env Linux** (NGC container `nvcr.io/nvidia/tensorflow:25.02-tf2-py3` + `TF_USE_LEGACY_KERAS=1`).
Windows env TF 2.15 **không đụng tới**.

Trước khi chạy:
1. `python prepare_dataset.py --root Dataset --predecode --num-frames 50` — tạo CSV + .npy (~30GB).
2. `python env_check.py` — xác nhận GPU + TF + MoviNet hoạt động.


## Section 1: Dependencies

Chạy trong NGC container hoặc venv Linux **sau khi** đã `TF_USE_LEGACY_KERAS=1`:
```
pip install tf-models-official==2.17.0 tf_keras==2.17.0 tfmot==0.8.0 \
            tf2onnx==1.16.1 onnxruntime-gpu decord opencv-python-headless \
            pandas scikit-learn seaborn
```


In [ ]:
# === Env setup — BẮT BUỘC set TRƯỚC khi import tensorflow ===
# FIX RTX 5080 (Blackwell): TF 2.21 quên preload libcusolver → GPU không được đăng ký.
# Preload toàn bộ lib NVIDIA từ pip wheels bằng ctypes TRƯỚC khi import TF.
import ctypes, glob, sysconfig
_sp = sysconfig.get_paths()["purelib"]
for _p in sorted(glob.glob(f"{_sp}/nvidia/*/lib/lib*.so.*")):
    try:
        ctypes.CDLL(_p, mode=ctypes.RTLD_GLOBAL)
    except OSError:
        pass

import os
os.environ.setdefault("CUDA_CACHE_MAXSIZE", "2147483648")  # cache JIT PTX→sm_120 (lần đầu chậm, sau nhanh)
os.environ["TF_USE_LEGACY_KERAS"] = "1"            # để movinet_model.MovinetClassifier (Keras 2) chạy trên TF 2.17
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
os.environ.setdefault("TF_CUDNN_DETERMINISTIC", "1")

import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import tensorflow as tf

# CRITICAL: `tensorflow_models` (official) PHẢI import TRƯỚC `tf_keras` để load weights đúng.
# GitHub issue tensorflow/models#13568. Không swap thứ tự này.
from official.projects.movinet.modeling import movinet, movinet_model

# Sau khi tensorflow_models đã import xong, mới dùng tf.keras (trỏ tới tf_keras qua TF_USE_LEGACY_KERAS).
from tensorflow.keras import mixed_precision

# Mixed precision BF16 — RTX 5080 Blackwell có native BF16 tensor core.
# KHÔNG dùng fp16: SE-block sigmoid×conv của MoviNet thường overflow fp16 range,
# còn bf16 có exponent range = fp32 → NaN rate ~0.
mixed_precision.set_global_policy("mixed_bfloat16")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow: {tf.__version__}")
print(f"Mixed precision policy: {mixed_precision.global_policy()}")
gpus = tf.config.list_physical_devices("GPU")
print(f"GPU devices: {gpus}")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
    details = tf.config.experimental.get_device_details(g)
    print(f"  {g.name} | cc={details.get('compute_capability')} | {details.get('device_name')}")


## ⚙️ Section 2: Configuration

### Tất cả hyperparameters được tổ chức trong 1 dict để dễ quản lý
### V3: Auto-detect classes, reduced epochs, MixUp support

In [ ]:
# ==================== MASTER CONFIGURATION V5 ====================
# Align với official MoViNet-A1 Stream Kinetics-600 recipe
# (movinet_a1_stream_k600_8x8.yaml từ tensorflow/models)
CONFIG = {
    # ==== Match official MoViNet-A1 Stream ====
    "resolution": 172,                          # official
    "num_frames": 50,                           # official (V4 dùng 24 — under-spec)
    "temporal_stride": 5,                       # official fixed stride
    "temporal_stride_jitter": 1,                # paper random_stride_range=1
    "num_classes": 2,
    "dropout_rate": 0.3,                        # official stream=0.2; 0.3 cho small-data
    "stochastic_depth_drop_rate": 0.2,          # official (V4 thiếu)
    "label_smoothing": 0.1,                     # official
    "weight_decay": 3e-5,                       # official L2 (V4 dùng 1e-4 quá mạnh với AdamW)
    "conv_type": "2plus1d",                     # match K600 checkpoint (official sẽ là 3d_2plus1d
                                                 # cho training throughput, nhưng convert ngược không
                                                 # support → giữ 2plus1d nhất quán training + export)
    "se_type": "2plus3d",                       # official
    "activation": "hard_swish",                 # official

    # ==== Optimizer — AdamW (user chose) ====
    "optimizer": "adamw",
    "stage1_lr": 1e-3,                          # head-only warmup LR
    "stage2_lr": 1e-4,                          # full fine-tune (10× nhỏ hơn)
    "optimizer_epsilon": 1e-7,                  # AdamW default
    "clip_norm": 1.0,
    "clip_value": 0.5,                          # extra NaN safety

    # ==== Training schedule — 2 stages đơn giản ====
    "batch_size": 8,                            # 16 bị OOM ở Stage 2 full fine-tune trên 16GB — activation backbone quá lớn
    "stage1_epochs": 4,                         # head-only
    "stage2_epochs": 18,                        # full fine-tune
    "warmup_epochs": 1,                         # ~2.5% total, official style
    "use_mixed_precision": True,
    "mp_policy": "mixed_bfloat16",              # BF16 native RTX 5080 Blackwell

    # ==== Loss ====
    "loss": "cce",                              # Categorical CE + label_smoothing (bỏ focal)

    # ==== EMA ====
    "use_ema": True,
    "ema_decay": 0.99,                          # Polyak paper (V4 dùng 0.9999 quá chậm)

    # ==== Augmentation — official-style (bỏ cutout + temporal reverse) ====
    "aug_hflip": True,
    "aug_random_resized_crop": True,
    "aug_area_min": 0.5,                        # relaxed cho small-data (official 0.08)
    "aug_area_max": 1.0,
    "aug_aspect_min": 0.75,                     # action recognition (official 0.5)
    "aug_aspect_max": 1.333,
    "aug_brightness": 0.2,
    "aug_contrast_min": 0.8, "aug_contrast_max": 1.2,
    "aug_saturation_min": 0.8, "aug_saturation_max": 1.2,
    "aug_hue": 0.05,
    "aug_mixup": True, "mixup_alpha": 0.2,      # small-data bonus (không có trong official)

    # ==== Eval ====
    "tta_crops": 10,                            # REPORT only, 1-crop cho model selection

    # ==== Reproducibility + paths ====
    "seed": 42, "deterministic": True,
    "oversample_weights": [0.5, 0.5],
    "class_names": ["normal", "steal"],
    "use_predecoded_npy": False,   # scoop: decode on-the-fly từ CSV
    "dataset_root": "dataset_scoop",
    "train_csv": "dataset_scoop/train.csv",
    "val_csv": "dataset_scoop/val.csv",
    "train_npy": "dataset_scoop/train_frames.npy",
    "train_labels_npy": "dataset_scoop/train_labels.npy",
    "val_npy": "dataset_scoop/val_frames.npy",
    "val_labels_npy": "dataset_scoop/val_labels.npy",

    # ==== Export ====
    "export_onnx": True, "onnx_opset": 17,
    "export_savedmodel": True,
}

RESOLUTION     = CONFIG["resolution"]
NUM_FRAMES     = CONFIG["num_frames"]
NUM_CLASSES    = CONFIG["num_classes"]
BATCH_SIZE     = CONFIG["batch_size"]
CLASS_NAMES    = CONFIG["class_names"]
CLASS_TO_IDX   = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS   = {i: c for c, i in CLASS_TO_IDX.items()}

print("\n=== CONFIG V5 (aligned với official MoViNet-A1 Stream recipe) ===")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


## 📂 Section 3: Data Pipeline (V3 — Fixed & Optimized)

### Cải tiến V3:
1. **Fix hue augmentation** — bỏ fake hue shift sai logic
2. **Auto-detect classes** — không cần hardcode class_labels
3. **Dynamic shuffle buffer** — buffer = dataset size
4. **Exact output_signature** — specify shape chính xác
5. **MixUp augmentation** — trên batch level
6. **Bỏ class_weight** — chỉ dùng oversampling (tránh double compensation)

In [ ]:
# ==================== V5: Data pipeline — fixed stride 5 + zero-pad (official-style) ====================
AUTOTUNE = tf.data.AUTOTUNE
_BASE_STRIDE = CONFIG["temporal_stride"]           # 5 — official
_STRIDE_JITTER = CONFIG["temporal_stride_jitter"]  # ±1 — paper random_stride_range

def resize_with_pad_np(img: np.ndarray, target_h: int, target_w: int) -> np.ndarray:
    """cv2-based resize_with_pad (bilinear, zero-padded)."""
    h, w = img.shape[:2]
    scale = min(target_h / h, target_w / w)
    new_h, new_w = int(round(h * scale)), int(round(w * scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    pad_top = (target_h - new_h) // 2
    pad_bottom = target_h - new_h - pad_top
    pad_left = (target_w - new_w) // 2
    pad_right = target_w - new_w - pad_left
    return cv2.copyMakeBorder(resized, pad_top, pad_bottom, pad_left, pad_right,
                              cv2.BORDER_CONSTANT, value=0)


def _sample_indices_train(total: int, n_frames: int) -> np.ndarray:
    """Fixed stride=5 ± jitter 1 + random start (official train recipe)."""
    stride = max(1, _BASE_STRIDE + int(np.random.randint(-_STRIDE_JITTER,
                                                          _STRIDE_JITTER + 1)))
    clip_len = n_frames * stride
    if total >= clip_len:
        start = np.random.randint(0, total - clip_len + 1)
        return np.arange(start, start + clip_len, stride)
    # Clip ngắn — zero-pad sẽ xử lý ở decode_video; trả index cho phần có thật.
    # Lấy đều trong phạm vi total (stride=1 hoặc nhỏ hơn).
    if total >= n_frames:
        return np.linspace(0, total - 1, n_frames).round().astype(np.int64)
    return np.arange(total, dtype=np.int64)  # decode_video sẽ zero-pad phần còn lại


def _sample_indices_uniform(total: int, n_frames: int) -> np.ndarray:
    """Uniform center-like sampling cho eval (1 crop). Zero-pad nếu thiếu."""
    if total <= 0:
        return np.arange(0, dtype=np.int64)
    if total >= n_frames:
        return np.linspace(0, total - 1, n_frames).round().astype(np.int64)
    return np.arange(total, dtype=np.int64)


def _sample_indices_multi_crop(total: int, n_frames: int, n_crops: int) -> np.ndarray:
    """10 uniform crop cho TTA, mỗi crop n_frames. Trả (n_crops, n_frames) với
    marker -1 cho slot cần zero-pad (khi clip ngắn hơn n_frames)."""
    if total < n_frames:
        idx = _sample_indices_uniform(total, n_frames)
        padded = np.full(n_frames, -1, dtype=np.int64)
        padded[:len(idx)] = idx
        return np.tile(padded[None, :], (n_crops, 1))
    stride = max(1, total // n_frames)
    clip_len = n_frames * stride
    if clip_len > total:
        clip_len = total
        stride = max(1, clip_len // n_frames)
    max_start = max(0, total - clip_len)
    starts = np.linspace(0, max_start, n_crops).round().astype(np.int64)
    return np.stack([np.linspace(s, s + clip_len - 1, n_frames).round().astype(np.int64)
                     for s in starts], axis=0)


def decode_video(path: str, n_frames: int, resolution: int,
                 training: bool = False) -> np.ndarray:
    """Decode 1 video → (n_frames, H, W, 3) float32 [0,1] RGB, resize_with_pad.
    Zero-pad tail cho clip ngắn hơn n_frames (match official frames_from_video_file)."""
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    indices = (_sample_indices_train(total, n_frames) if training
               else _sample_indices_uniform(total, n_frames))

    out = np.zeros((n_frames, resolution, resolution, 3), dtype=np.float32)
    if total <= 0 or len(indices) == 0:
        cap.release()
        return out

    want = sorted(set(int(i) for i in indices))
    cache: dict[int, np.ndarray] = {}
    idx = 0
    wi = 0
    while cap.isOpened() and wi < len(want):
        if idx == want[wi]:
            ok, frame = cap.read()
            if not ok:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = resize_with_pad_np(frame, resolution, resolution)
            cache[idx] = frame.astype(np.float32) / 255.0
            wi += 1
            idx += 1
        else:
            cap.grab()
            idx += 1
    cap.release()

    # Zero-pad: chỉ ghi vào slot có frame thật, còn lại giữ zero
    for i, fidx in enumerate(indices):
        if i >= n_frames:
            break
        if int(fidx) in cache:
            out[i] = cache[int(fidx)]
    return out


def decode_video_multi_crop(path: str, n_frames: int, resolution: int,
                             n_crops: int) -> np.ndarray:
    """Decode 10 uniform crop cho TTA report → (n_crops, n_frames, H, W, 3). Zero-pad."""
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    all_idx = _sample_indices_multi_crop(total, n_frames, n_crops)
    out = np.zeros((n_crops, n_frames, resolution, resolution, 3), dtype=np.float32)
    if total <= 0:
        cap.release()
        return out

    want = sorted(set(int(i) for i in all_idx.flatten() if i >= 0))
    cache: dict[int, np.ndarray] = {}
    idx = 0
    wi = 0
    while cap.isOpened() and wi < len(want):
        if idx == want[wi]:
            ok, frame = cap.read()
            if not ok:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = resize_with_pad_np(frame, resolution, resolution)
            cache[idx] = frame.astype(np.float32) / 255.0
            wi += 1
            idx += 1
        else:
            cap.grab()
            idx += 1
    cap.release()

    for ci in range(n_crops):
        for fi in range(n_frames):
            fidx = int(all_idx[ci, fi])
            if fidx >= 0 and fidx in cache:
                out[ci, fi] = cache[fidx]
    return out


# ==================== V5 Augmentation — official-style (graph-safe) ====================
# Các op điều kiện phải dùng tf.cond chứ không Python `if` trên tensor.
# V5 thay đổi so với V4:
#   - Bỏ cutout (không có trong official)
#   - Bỏ class-conditional temporal reverse (không có trong official)
#   - Thay pad-then-crop cố định → random resized crop (official aug_type equivalent)

_RES = CONFIG["resolution"]
_AREA_MIN = CONFIG["aug_area_min"]
_AREA_MAX = CONFIG["aug_area_max"]
_ASP_MIN  = CONFIG["aug_aspect_min"]
_ASP_MAX  = CONFIG["aug_aspect_max"]

def _flip(clip):
    return tf.image.flip_left_right(clip)

def _random_resized_crop(clip):
    """Random resized crop toàn clip — official aug_min_area_ratio/max_area_ratio style.
    Cùng crop box áp dụng cho mọi frame để temporal coherence."""
    shape = tf.shape(clip)
    h = tf.cast(shape[1], tf.float32)
    w = tf.cast(shape[2], tf.float32)
    area = h * w

    # Sample area và aspect ratio
    target_area = area * tf.random.uniform([], _AREA_MIN, _AREA_MAX)
    aspect = tf.random.uniform([], _ASP_MIN, _ASP_MAX)
    crop_w = tf.sqrt(target_area * aspect)
    crop_h = tf.sqrt(target_area / aspect)

    # Clip về trong ảnh
    crop_h = tf.minimum(crop_h, h)
    crop_w = tf.minimum(crop_w, w)
    crop_h_i = tf.cast(crop_h, tf.int32)
    crop_w_i = tf.cast(crop_w, tf.int32)

    # Random offset (cùng cho mọi frame)
    max_y = tf.maximum(1, shape[1] - crop_h_i)
    max_x = tf.maximum(1, shape[2] - crop_w_i)
    y0 = tf.random.uniform([], 0, max_y, dtype=tf.int32)
    x0 = tf.random.uniform([], 0, max_x, dtype=tf.int32)

    cropped = clip[:, y0:y0 + crop_h_i, x0:x0 + crop_w_i, :]
    resized = tf.image.resize(cropped, [_RES, _RES],
                               method=tf.image.ResizeMethod.BILINEAR, antialias=True)
    return resized


def _augment_clip(clip: tf.Tensor, class_idx: tf.Tensor) -> tf.Tensor:
    """V5 augmentation (graph-safe) — bỏ cutout + temporal reverse theo official."""
    # 1. Random resized crop (spatial aug chính — regularizer mạnh cho small-data)
    clip = _random_resized_crop(clip)

    # 2. Horizontal flip p=0.5
    clip = tf.cond(tf.less(tf.random.uniform([]), 0.5),
                   lambda: _flip(clip), lambda: clip)

    # 3. Color jitter (mild, same cho mọi frame qua tf.image.random_*)
    clip = tf.image.random_brightness(clip, max_delta=CONFIG["aug_brightness"])
    clip = tf.image.random_contrast(clip, lower=CONFIG["aug_contrast_min"],
                                          upper=CONFIG["aug_contrast_max"])
    clip = tf.image.random_saturation(clip, lower=CONFIG["aug_saturation_min"],
                                             upper=CONFIG["aug_saturation_max"])
    clip = tf.image.random_hue(clip, max_delta=CONFIG["aug_hue"])
    clip = tf.clip_by_value(clip, 0.0, 1.0)

    return clip


def build_dataset_from_npy(x_path: str, y_path: str, training: bool,
                           oversample_weights=None) -> tf.data.Dataset:
    """Tải pre-decoded .npy + wrap thành tf.data.Dataset (nhanh nhất, dùng 120GB RAM)."""
    X = np.load(x_path, mmap_mode="r")
    y = np.load(y_path)
    print(f"  Loaded {x_path}: X.shape={X.shape} y.shape={y.shape}")

    if training and oversample_weights is not None:
        # Tách theo class rồi sample_from_datasets
        per_class = []
        for ci in range(NUM_CLASSES):
            mask = (y == ci)
            Xi = np.asarray(X[mask])    # materialize (small dataset)
            yi = y[mask]
            d = tf.data.Dataset.from_tensor_slices((Xi, yi))
            d = d.shuffle(len(yi), seed=SEED, reshuffle_each_iteration=True).repeat()
            per_class.append(d)
        ds = tf.data.Dataset.sample_from_datasets(per_class, weights=oversample_weights, seed=SEED)
    else:
        ds = tf.data.Dataset.from_tensor_slices((np.asarray(X), y))
        if training:
            ds = ds.shuffle(len(y), seed=SEED, reshuffle_each_iteration=True)

    if training:
        ds = ds.map(lambda clip, lbl: (_augment_clip(clip, lbl), lbl),
                    num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda clip, lbl: (clip, tf.one_hot(tf.cast(lbl, tf.int32), NUM_CLASSES)),
                num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=training).prefetch(AUTOTUNE)
    if not training:
        ds = ds.cache()
    return ds


def build_dataset_from_csv(csv_path: str, training: bool,
                            oversample_weights=None) -> tf.data.Dataset:
    """Fallback khi chưa có .npy — decode trên-the-fly."""
    df = pd.read_csv(csv_path)
    paths = df["path"].tolist()
    labels = np.array([CLASS_TO_IDX[c] for c in df["class"]], dtype=np.int32)
    print(f"  {csv_path}: {len(paths)} videos")

    def _decode(path_bytes, label, is_train):
        path = path_bytes.numpy().decode("utf-8") if hasattr(path_bytes, "numpy") else path_bytes
        clip = decode_video(path, NUM_FRAMES, RESOLUTION, training=bool(is_train))
        return clip.astype(np.float32), np.int32(label)

    def _tf_decode(path, label, is_train):
        clip, lbl = tf.py_function(
            _decode, [path, label, is_train], [tf.float32, tf.int32])
        clip.set_shape([NUM_FRAMES, RESOLUTION, RESOLUTION, 3])
        lbl.set_shape([])
        return clip, lbl

    if training and oversample_weights is not None:
        per_class = []
        for ci in range(NUM_CLASSES):
            mask = (labels == ci)
            pi = [p for p, m in zip(paths, mask) if m]
            li = labels[mask]
            d = tf.data.Dataset.from_tensor_slices((pi, li))
            d = d.shuffle(len(li), seed=SEED, reshuffle_each_iteration=True).repeat()
            d = d.map(lambda p, l: _tf_decode(p, l, True), num_parallel_calls=AUTOTUNE)
            per_class.append(d)
        ds = tf.data.Dataset.sample_from_datasets(per_class, weights=oversample_weights, seed=SEED)
    else:
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(len(labels), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(lambda p, l: _tf_decode(p, l, training), num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.map(lambda clip, lbl: (_augment_clip(clip, lbl), lbl),
                    num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda clip, lbl: (clip, tf.one_hot(tf.cast(lbl, tf.int32), NUM_CLASSES)),
                num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=training).prefetch(AUTOTUNE)
    if not training:
        ds = ds.cache()
    return ds


print("✓ Data pipeline V4 defined (stride sampler, reflect-pad, augmentation, CSV/NPY sources)")


In [ ]:
# ==================== Build train/val datasets ====================
# Ưu tiên .npy (pre-decoded bởi prepare_dataset.py, tận dụng 120GB RAM).
# Fallback: đọc CSV + decode on-the-fly.

def _file_ok(p: str) -> bool:
    return os.path.exists(p) and os.path.getsize(p) > 0

if (CONFIG["use_predecoded_npy"]
        and _file_ok(CONFIG["train_npy"]) and _file_ok(CONFIG["train_labels_npy"])
        and _file_ok(CONFIG["val_npy"])   and _file_ok(CONFIG["val_labels_npy"])):
    print("→ Dùng pre-decoded .npy (nhanh nhất)")
    train_ds = build_dataset_from_npy(CONFIG["train_npy"], CONFIG["train_labels_npy"],
                                       training=True,
                                       oversample_weights=CONFIG["oversample_weights"])
    val_ds   = build_dataset_from_npy(CONFIG["val_npy"],   CONFIG["val_labels_npy"],
                                       training=False)
else:
    print("→ Chưa có .npy, đọc CSV + decode on-the-fly. Chạy prepare_dataset.py --predecode để tăng tốc.")
    train_ds = build_dataset_from_csv(CONFIG["train_csv"], training=True,
                                       oversample_weights=CONFIG["oversample_weights"])
    val_ds   = build_dataset_from_csv(CONFIG["val_csv"], training=False)

# Sanity check: 1 batch
for xb, yb in train_ds.take(1):
    print(f"  train batch: x={xb.shape} y={yb.shape}, x.range=[{tf.reduce_min(xb):.3f}, {tf.reduce_max(xb):.3f}]")
    assert tf.reduce_all(tf.math.is_finite(xb)), "Non-finite in train batch"
for xb, yb in val_ds.take(1):
    print(f"  val batch:   x={xb.shape} y={yb.shape}")


In [ ]:
# (Cell này trong v3 dùng FrameGenerator + pathlib để setup paths.
#  V4 đã chuyển sang CSV/NPY-driven pipeline ở cell trước, nên cell này là no-op.)
print("✓ Dataset đã build ở cell trước (CSV/NPY pipeline).")


In [ ]:
# ==================== V4: MixUp batch-level (graph-safe, không dùng compat.v1) ====================
def _beta_sample(alpha):
    """Beta(alpha, alpha) sample qua 2 Gamma: X/(X+Y), X,Y ~ Gamma(alpha, 1)."""
    g1 = tf.random.gamma([], alpha, dtype=tf.float32)
    g2 = tf.random.gamma([], alpha, dtype=tf.float32)
    return g1 / (g1 + g2 + 1e-8)

def mixup_batch(x, y, alpha=CONFIG["mixup_alpha"]):
    """MixUp 2 sample bất kỳ trong cùng batch."""
    batch_size = tf.shape(x)[0]
    lam = _beta_sample(alpha)
    lam = tf.maximum(lam, 1.0 - lam)  # lam >= 0.5 để stable
    lam_x = tf.cast(lam, x.dtype)
    lam_y = tf.cast(lam, y.dtype)
    idx = tf.random.shuffle(tf.range(batch_size))
    x2 = tf.gather(x, idx)
    y2 = tf.gather(y, idx)
    x_mix = lam_x * x + (1.0 - lam_x) * x2
    y_mix = lam_y * y + (1.0 - lam_y) * y2
    return x_mix, y_mix

def apply_mixup(ds, alpha=CONFIG["mixup_alpha"]):
    return ds.map(lambda x, y: mixup_batch(x, y, alpha), num_parallel_calls=AUTOTUNE)

train_ds_mixed = apply_mixup(train_ds, alpha=CONFIG["mixup_alpha"])
print("✓ MixUp applied to train_ds_mixed")


## 🏗️ Section 4: Build MoViNet Stream Model

### V3: Simplified freeze logic

In [ ]:
# ==================== V5: Build MoviNet-A1 Stream + load Kinetics-600 ====================
# Thay đổi so với V4:
#   + stochastic_depth_drop_rate=0.2 (official, V4 thiếu — regularizer quan trọng)
#   + dropout_rate=0.3 (giảm từ 0.4, gần hơn official stream=0.2)
#
# Ghi chú conv_type: K600 pretrained checkpoint từ TF Garden build với 'conv_type=2plus1d'.
# Chuyển '2plus1d' → '3d_2plus1d' cho training không được convert_3d_2plus1d.py hỗ trợ
# (script chỉ đi một chiều). Do đó V5 giữ '2plus1d' cho cả training và export — bỏ
# throughput bonus của 3d_2plus1d để đổi lấy stability + không phải map weights thủ công.
# Nếu sau này K600 3d_2plus1d checkpoint được publish, ta có thể chuyển qua.
import tarfile, urllib.request

MODEL_ID = "a1"
WEIGHT_URL = f"https://storage.googleapis.com/tf_model_garden/vision/movinet/movinet_{MODEL_ID}_stream.tar.gz"
CKPT_DIR = f"movinet_{MODEL_ID}_stream"
TAR_FILE = f"movinet_{MODEL_ID}_stream.tar.gz"

print("=" * 80)
print("1️⃣ Creating MoViNet Stream backbone...")
backbone = movinet.Movinet(
    model_id=MODEL_ID,
    causal=True,
    conv_type="2plus1d",                                              # match K600 ckpt
    se_type=CONFIG["se_type"],                                         # 2plus3d
    activation=CONFIG["activation"],                                   # hard_swish
    gating_activation="hard_sigmoid",
    use_positional_encoding=False,
    use_external_states=False,
    stochastic_depth_drop_rate=CONFIG["stochastic_depth_drop_rate"],   # 0.2 (official, mới)
)
print(f"   ✓ Backbone: MoViNet-{MODEL_ID.upper()} Stream (stochastic_depth=0.2)")

print("\n2️⃣ Downloading Kinetics-600 weights...")
if not os.path.exists(CKPT_DIR):
    print(f"   Downloading {WEIGHT_URL} ...")
    urllib.request.urlretrieve(WEIGHT_URL, TAR_FILE)
    with tarfile.open(TAR_FILE) as tar:
        tar.extractall()
    os.remove(TAR_FILE)
checkpoint_path = tf.train.latest_checkpoint(CKPT_DIR)
print(f"   ✓ checkpoint: {checkpoint_path}")

print("\n3️⃣ Loading pretrained (Kinetics-600, 600 classes)...")
pretrained = movinet_model.MovinetClassifier(
    backbone=backbone, num_classes=600, output_states=True)
pretrained.build([1, NUM_FRAMES, RESOLUTION, RESOLUTION, 3])
ckpt = tf.train.Checkpoint(model=pretrained)
ckpt.restore(checkpoint_path).assert_existing_objects_matched()
print("   ✓ Kinetics-600 weights loaded")

print("\n4️⃣ Extracting trained backbone for custom head...")
trained_backbone = pretrained.backbone

print("\n5️⃣ Creating target model (NUM_CLASSES={NUM_CLASSES}, dropout=0.3)...")
model = movinet_model.MovinetClassifier(
    backbone=trained_backbone,
    num_classes=NUM_CLASSES,
    dropout_rate=CONFIG["dropout_rate"],
)
model.build([BATCH_SIZE, NUM_FRAMES, RESOLUTION, RESOLUTION, 3])

# Freeze backbone cho Stage 1 (head-only)
model.backbone.trainable = False
n_trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
n_total = sum(np.prod(v.shape) for v in model.variables)
print(f"\n   ✓ Model built. Stage-1 trainable: {n_trainable:,} / total: {n_total:,}")


## 🛡️ Section 5: NaN-Safe Training Utilities

### V3: WarmupCosineDecaySchedule now ACTUALLY USED via LearningRateScheduler callback

In [ ]:
# ==================== V5: CCE + EMA (0.99) + NaN-skip train_step + WarmupCosine ====================
# Thay focal loss → CategoricalCrossentropy với label_smoothing (official recipe).
# Focal là over-engineer cho imbalance 2.3× (mild); oversampling một mình đã đủ.

def build_loss():
    """Official MoViNet loss: CCE from logits + label_smoothing 0.1."""
    return tf.keras.losses.CategoricalCrossentropy(
        from_logits=True,
        label_smoothing=CONFIG["label_smoothing"],
    )


class WarmupCosineDecaySchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, warmup_steps, total_steps, min_lr=1e-6):
        super().__init__()
        self.base_lr = float(base_lr); self.warmup_steps = int(warmup_steps)
        self.total_steps = int(total_steps); self.min_lr = float(min_lr)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        ws = tf.cast(self.warmup_steps, tf.float32)
        ts = tf.cast(self.total_steps, tf.float32)
        warmup_lr = self.base_lr * (step / tf.maximum(ws, 1.0))
        decay_progress = (step - ws) / tf.maximum(ts - ws, 1.0)
        decay_progress = tf.clip_by_value(decay_progress, 0.0, 1.0)
        cosine_lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * \
                    (1.0 + tf.cos(tf.constant(np.pi) * decay_progress))
        return tf.where(step < ws, warmup_lr, cosine_lr)


class EMACallback(tf.keras.callbacks.Callback):
    """Exponential Moving Average weights. Apply at end of training.
    V5 dùng decay=0.99 (Polyak paper) thay 0.9999 — phù hợp với training ngắn ~300 steps."""
    def __init__(self, decay=0.99):
        super().__init__()
        self.decay = decay
        self.shadow = None

    def on_train_begin(self, logs=None):
        self.shadow = [tf.Variable(w, trainable=False) for w in self.model.weights]

    def on_train_batch_end(self, batch, logs=None):
        if self.shadow is None:
            return
        d = self.decay
        for s, w in zip(self.shadow, self.model.weights):
            s.assign(d * s + (1.0 - d) * w)

    def on_train_end(self, logs=None):
        if self.shadow is None:
            return
        for w, s in zip(self.model.weights, self.shadow):
            w.assign(s)
        print(f"✓ EMA weights applied at train end (decay={self.decay})")


class GradientMonitorCallback(tf.keras.callbacks.Callback):
    def __init__(self, log_every=20, stop_after_epoch=2):
        super().__init__()
        self.log_every = log_every
        self.stop_after_epoch = stop_after_epoch
        self.epoch = 0

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch = epoch

    def on_train_batch_end(self, batch, logs=None):
        if self.epoch >= self.stop_after_epoch:
            return
        if logs and batch % self.log_every == 0:
            loss = logs.get("loss")
            if loss is not None and (not np.isfinite(loss)):
                print(f"  ⚠ Non-finite loss at batch {batch}: {loss}")


def make_nan_safe_model(model):
    """Override train_step để skip optimizer step khi grads có NaN/Inf."""
    base_train_step = model.train_step

    def safe_train_step(data):
        x, y = data
        with tf.GradientTape() as tape:
            y_pred = model(x, training=True)
            loss = model.compiled_loss(y, y_pred, regularization_losses=model.losses)
        trainable = model.trainable_variables
        grads = tape.gradient(loss, trainable)
        any_nan = tf.constant(False)
        for g in grads:
            if g is not None:
                any_nan = tf.logical_or(
                    any_nan,
                    tf.logical_or(tf.reduce_any(tf.math.is_nan(g)),
                                  tf.reduce_any(tf.math.is_inf(g))),
                )
        def apply():
            model.optimizer.apply_gradients(zip(grads, trainable))
            return tf.constant(0.0)
        def skip():
            tf.print("⚠ NaN/Inf grad — skip optimizer step")
            return tf.constant(1.0)
        tf.cond(any_nan, skip, apply)
        model.compiled_metrics.update_state(y, y_pred)
        return {m.name: m.result() for m in model.metrics} | {"loss": loss}

    model.train_step = safe_train_step
    return model


print("✓ V5 utilities defined: build_loss (CCE+LS), WarmupCosineDecaySchedule, EMACallback (0.99), GradientMonitorCallback, make_nan_safe_model")


## 🚀 Section 6: Three-Stage Progressive Training

### V3 Changes:
- **AdamW** thay vì Adam (weight_decay hoạt động)
- **WarmupCosineDecay** qua LearningRateScheduler (thay vì ReduceLROnPlateau)
- **Bỏ class_weight** (chỉ dùng oversampling)
- **MixUp dataset** cho training
- **Giảm epochs** (10+15+10 = 35 thay vì 60)
- **Giảm patience** (7 thay vì 12)

```
Stage 1: [FROZEN backbone] → [TRAIN head]        ← Learn task-specific mapping
Stage 2: [FROZEN early] → [TRAIN top 40%] → head ← Adapt high-level features  
Stage 3: [TRAIN ALL layers]                       ← Fine-tune everything
```

```
Stage 1: lr=1e-3 + warmup 2 epochs + cosine decay  (head random → need high LR)
Stage 2: lr=5e-5 + warmup 1 epoch + cosine decay   (pretrained → low LR)
Stage 3: lr=1e-5 + warmup 1 epoch + cosine decay   (full model → very careful)
```

In [ ]:
# ==================== STAGE 1: Head-only training (backbone frozen) ====================
# V5 simplifies V4 (bỏ focal, AdamW với weight_decay 3e-5, 4 epochs + 1 warmup)
print("=" * 80)
print("STAGE 1: Head-only training (4 epochs + 1 warmup)")
print("=" * 80)

model.backbone.trainable = False
n_trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f"Trainable params: {n_trainable:,}")

# Compute steps_per_epoch — train_ds_mixed infinite vì sample_from_datasets().repeat()
_train_df = pd.read_csv(CONFIG["train_csv"])
_max_cls = int(_train_df["class"].value_counts().max())
STEPS_PER_EPOCH = max(10, (_max_cls * NUM_CLASSES) // BATCH_SIZE)
print(f"Steps per epoch: {STEPS_PER_EPOCH} (max_class_count={_max_cls})")

# V5: warmup_epochs = 1 (official style ~2.5% total), không per-stage warmup dài
_WARMUP_STEPS = STEPS_PER_EPOCH * CONFIG["warmup_epochs"]
total_steps_stage1 = STEPS_PER_EPOCH * CONFIG["stage1_epochs"]

lr_schedule = WarmupCosineDecaySchedule(
    base_lr=CONFIG["stage1_lr"],            # 1e-3 for head-only
    warmup_steps=_WARMUP_STEPS,
    total_steps=total_steps_stage1,
)

# AdamW (user choice) — weight_decay decoupled, tốt cho fine-tune
optimizer = tf.keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=CONFIG["weight_decay"],     # 3e-5 (match official L2)
    clipnorm=CONFIG["clip_norm"],
    clipvalue=CONFIG["clip_value"],
    epsilon=CONFIG["optimizer_epsilon"],
)

loss_fn = build_loss()                       # CCE + label_smoothing 0.1

model.compile(optimizer=optimizer, loss=loss_fn,
              metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")])
make_nan_safe_model(model)

callbacks_stage1 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=False),
    tf.keras.callbacks.ModelCheckpoint("best_movinet_scoop_stage1.weights.h5",
                                        monitor="val_accuracy", mode="max",
                                        save_best_only=True, save_weights_only=True),
    tf.keras.callbacks.CSVLogger("training_stage1.csv"),
    GradientMonitorCallback(log_every=20, stop_after_epoch=1),
]
if CONFIG["use_ema"]:
    callbacks_stage1.append(EMACallback(decay=CONFIG["ema_decay"]))   # 0.99

print(f"Stage 1 ready. LR={CONFIG['stage1_lr']}, epochs={CONFIG['stage1_epochs']}, warmup={CONFIG['warmup_epochs']}")


In [ ]:
# Pre-training sanity check
print("🧪 Pre-training sanity check...")
for frames, labels in train_ds_mixed.take(1):
    pred = model(frames, training=False)
    print(f"   Input shape: {frames.shape}")
    print(f"   Input range: [{tf.reduce_min(frames):.3f}, {tf.reduce_max(frames):.3f}]")
    print(f"   Output shape: {pred.shape}")
    print(f"   Output range: [{tf.reduce_min(pred):.3f}, {tf.reduce_max(pred):.3f}]")
    print(f"   NaN in input: {tf.reduce_any(tf.math.is_nan(frames)).numpy()}")
    print(f"   NaN in output: {tf.reduce_any(tf.math.is_nan(pred)).numpy()}")
    print(f"   Labels shape: {labels.shape}")
    
    # Test loss computation
    test_loss = loss_fn(labels, pred)
    print(f"   Test loss: {test_loss:.4f}")
    print(f"   NaN in loss: {tf.math.is_nan(test_loss).numpy()}")

print("✅ Sanity check passed!")

In [ ]:
# Fit Stage 1 — head only
history_stage1 = model.fit(
    train_ds_mixed,
    validation_data=val_ds,
    epochs=CONFIG["stage1_epochs"],
    steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=callbacks_stage1,
    verbose=1,
)


In [ ]:
# ==================== STAGE 2: Full fine-tune (V5 simplifies V4 3-stage → 2-stage) ====================
print("=" * 80)
print("STAGE 2: Full fine-tune (18 epochs, LR 1e-4)")
print("=" * 80)

if os.path.exists("best_movinet_scoop_stage1.weights.h5"):
    model.load_weights("best_movinet_scoop_stage1.weights.h5")
    print("✓ Loaded best Stage 1 weights")

# Unfreeze ALL — end-to-end fine-tune theo official style
for layer in model.layers:
    layer.trainable = True
n_trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f"All layers unfrozen. Trainable params: {n_trainable:,}")

total_steps_stage2 = STEPS_PER_EPOCH * CONFIG["stage2_epochs"]

lr_schedule = WarmupCosineDecaySchedule(
    base_lr=CONFIG["stage2_lr"],          # 1e-4 (10× smaller than Stage 1)
    warmup_steps=0,                        # no warmup — smooth cosine từ head-only weights
    total_steps=total_steps_stage2,
)
optimizer = tf.keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=CONFIG["weight_decay"],
    clipnorm=CONFIG["clip_norm"],
    clipvalue=CONFIG["clip_value"],
    epsilon=CONFIG["optimizer_epsilon"],
)
model.compile(optimizer=optimizer, loss=loss_fn,
              metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")])
make_nan_safe_model(model)

callbacks_stage2 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=False),
    tf.keras.callbacks.ModelCheckpoint("best_movinet_scoop_stage2.weights.h5",
                                        monitor="val_accuracy", mode="max",
                                        save_best_only=True, save_weights_only=True),
    tf.keras.callbacks.CSVLogger("training_stage2.csv"),
]
if CONFIG["use_ema"]:
    callbacks_stage2.append(EMACallback(decay=CONFIG["ema_decay"]))

print(f"Stage 2 ready. LR={CONFIG['stage2_lr']} (10× smaller than Stage 1), epochs={CONFIG['stage2_epochs']}")


In [ ]:
# Fit Stage 2 — full fine-tune
history_stage2 = model.fit(
    train_ds_mixed,
    validation_data=val_ds,
    epochs=CONFIG["stage2_epochs"],
    steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=callbacks_stage2,
    verbose=1,
)


In [ ]:
# V5 simplifies V4 3-stage → 2-stage. Stage 3 không còn — Stage 2 đã full fine-tune end-to-end.
print("V5: Stage 3 đã được gộp vào Stage 2 (full fine-tune end-to-end). Skip.")


In [ ]:
# V5: Skip (Stage 3 đã gộp vào Stage 2)
history_stage3 = None


## 📊 Section 7: Comprehensive Evaluation

In [ ]:
# ==================== Load best model across stages + TTA evaluation ====================
stage_weights = {
    1: "best_movinet_scoop_stage1.weights.h5",
    2: "best_movinet_scoop_stage2.weights.h5",
}
best_stage = None
best_val_acc = -1.0
for stage, path in stage_weights.items():
    if not os.path.exists(path):
        continue
    print(f"Evaluating {path}...")
    try:
        model.load_weights(path)
        res = model.evaluate(val_ds, verbose=0, return_dict=True)
        print(f"  Stage {stage}: {res}")
        if res["accuracy"] > best_val_acc:
            best_val_acc = res["accuracy"]
            best_stage = stage
    except Exception as e:
        print(f"  Error loading stage {stage}: {e}")

if best_stage is not None:
    model.load_weights(stage_weights[best_stage])
    print(f"\n✓ Best stage = {best_stage} with val_acc = {best_val_acc:.4f}")
else:
    print("\n⚠ No best weights found. Using current model.")


In [ ]:
# ==================== Plot training history (V5: 2 stages) ====================
histories = {}
for stage in (1, 2):
    name = f"history_stage{stage}"
    if name in globals() and globals()[name] is not None:
        histories[stage] = globals()[name]

if histories:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for stage, h in histories.items():
        ep = np.arange(1, len(h.history["loss"]) + 1)
        axes[0].plot(ep, h.history["loss"], label=f"S{stage} train")
        axes[0].plot(ep, h.history.get("val_loss", []), "--", label=f"S{stage} val")
        axes[1].plot(ep, h.history.get("accuracy", []), label=f"S{stage} train")
        axes[1].plot(ep, h.history.get("val_accuracy", []), "--", label=f"S{stage} val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(True)
    axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(True)
    plt.tight_layout(); plt.savefig("training_history_v4.png", dpi=120); plt.show()


In [ ]:
# ==================== Classification report (1-crop + TTA 10-crop) ====================
def predict_1crop(ds):
    y_true_list, y_pred_list = [], []
    for xb, yb in ds:
        logits = model(xb, training=False)
        # logits là dict (MovinetClassifier output_states) hoặc tensor
        if isinstance(logits, dict):
            logits = logits.get("logits", list(logits.values())[0])
        y_pred = tf.argmax(tf.nn.softmax(logits, axis=-1), axis=-1).numpy()
        y_true = tf.argmax(yb, axis=-1).numpy()
        y_true_list.append(y_true); y_pred_list.append(y_pred)
    return np.concatenate(y_true_list), np.concatenate(y_pred_list)

def predict_tta(csv_path, n_crops):
    df = pd.read_csv(csv_path)
    y_true = np.array([CLASS_TO_IDX[c] for c in df["class"]], dtype=np.int32)
    y_pred = np.zeros(len(df), dtype=np.int32)
    for i, path in enumerate(df["path"].tolist()):
        clips = decode_video_multi_crop(path, NUM_FRAMES, RESOLUTION, n_crops).astype(np.float32)
        logits = model(clips, training=False)
        if isinstance(logits, dict):
            logits = logits.get("logits", list(logits.values())[0])
        probs = tf.nn.softmax(logits, axis=-1).numpy()
        y_pred[i] = int(np.argmax(probs.mean(axis=0)))
        if (i + 1) % 25 == 0:
            print(f"  TTA {i + 1}/{len(df)}")
    return y_true, y_pred

print("=== 1-crop val accuracy ===")
yt1, yp1 = predict_1crop(val_ds)
print(classification_report(yt1, yp1, target_names=CLASS_NAMES, digits=4))

if CONFIG["tta_crops"] > 1 and os.path.exists(CONFIG["val_csv"]):
    print(f"\n=== TTA {CONFIG['tta_crops']}-crop val accuracy ===")
    yt_tta, yp_tta = predict_tta(CONFIG["val_csv"], CONFIG["tta_crops"])
    print(classification_report(yt_tta, yp_tta, target_names=CLASS_NAMES, digits=4))
    tta_f1 = f1_score(yt_tta, yp_tta, average="macro")
    print(f"Macro-F1 (TTA): {tta_f1:.4f}")


In [ ]:
# ==================== Confusion matrix ====================
cm = confusion_matrix(yt1, yp1)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion Matrix (1-crop)")
plt.tight_layout(); plt.savefig("confusion_matrix_v4.png", dpi=120); plt.show()


In [ ]:
# ==================== Confidence histogram ====================
def predict_probs(ds):
    probs_list = []; true_list = []
    for xb, yb in ds:
        logits = model(xb, training=False)
        if isinstance(logits, dict):
            logits = logits.get("logits", list(logits.values())[0])
        probs = tf.nn.softmax(logits, axis=-1).numpy()
        probs_list.append(probs); true_list.append(tf.argmax(yb, axis=-1).numpy())
    return np.concatenate(probs_list), np.concatenate(true_list)

probs, yt = predict_probs(val_ds)
confs = probs[np.arange(len(yt)), yt]
correct = probs.argmax(axis=1) == yt
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(confs[correct], bins=20, alpha=0.7, label="correct")
ax.hist(confs[~correct], bins=20, alpha=0.7, label="wrong")
ax.set_xlabel("Confidence on true class"); ax.set_ylabel("Count"); ax.legend()
ax.set_title(f"Correct: {correct.mean()*100:.2f}% | Mean conf (correct): {confs[correct].mean():.3f}")
plt.tight_layout(); plt.savefig("confidence_v4.png", dpi=120); plt.show()


## 🎥 Section 8: Streaming Inference Model

In [ ]:
# ==================== Build streaming inference model (frame-by-frame) ====================
print("=" * 80)
print("Building streaming inference model...")
print("=" * 80)

# FIX: streaming state buffers là float32 nhưng mixed_bfloat16 khiến activation là bf16
#      → "AddV2: float32 does not match bfloat16" ở x + buffer.
#      Dựng + export streaming dưới float32 (inference từng frame, không cần bf16;
#      float32 còn chuẩn hơn cho export). Weights train ra lưu ở float32 nên load khớp.
from tensorflow.keras import mixed_precision as _mp
_mp.set_global_policy("float32")
print("  policy → float32 cho streaming/export")

streaming_backbone = movinet.Movinet(
    model_id=MODEL_ID,
    causal=True,
    conv_type="2plus1d",
    se_type="2plus3d",
    activation="hard_swish",
    gating_activation="hard_sigmoid",
    use_positional_encoding=False,
    use_external_states=True,  # ← stream mode
)

streaming_model = movinet_model.MovinetClassifier(
    backbone=streaming_backbone,
    num_classes=NUM_CLASSES,
    output_states=True,
)
streaming_model.build([1, 1, RESOLUTION, RESOLUTION, 3])

# Tạo đầy đủ variables cho graph external-state trước khi load_weights.
# Nếu chỉ build bằng shape, một số layer/state trong MoViNet có thể chưa được materialize.
_dummy_frame = tf.zeros([1, 1, RESOLUTION, RESOLUTION, 3], dtype=tf.float32)
_dummy_states = streaming_model.init_states(tf.shape(_dummy_frame))
_ = streaming_model({**_dummy_states, "image": _dummy_frame}, training=False)
del _dummy_frame, _dummy_states, _

# Load best weights (stage với val_acc cao nhất)
best_path = stage_weights.get(best_stage) if best_stage else None
if best_path and os.path.exists(best_path):
    streaming_model.load_weights(best_path)
    print(f"✓ Streaming weights ← {best_path}")
else:
    print("⚠ Không tìm thấy best weights, streaming model dùng random init.")

print(f"  Input:  (1, 1, {RESOLUTION}, {RESOLUTION}, 3) — single frame")
print(f"  Output: (logits, states)")


In [ ]:
# ==================== Test streaming inference on val set ====================
print("Testing streaming inference...")

def _unpack_streaming_output(output):
    if isinstance(output, tuple) or isinstance(output, list):
        return output[0], output[1]
    if isinstance(output, dict):
        logits = output.get("logits", output.get("classifier_head"))
        if logits is None:
            logits = next(v for k, v in output.items() if "state" not in k.lower())
        states = {k: v for k, v in output.items() if k != "logits" and k != "classifier_head"}
        return logits, states
    raise TypeError(f"Unexpected streaming output type: {type(output)}")

correct = 0; total = 0
for xb, yb in val_ds.take(2):
    for vid_idx in range(min(3, xb.shape[0])):
        video = xb[vid_idx].numpy()
        true_label = int(np.argmax(yb[vid_idx].numpy()))
        states = streaming_model.init_states(tf.constant([1, 1, RESOLUTION, RESOLUTION, 3], dtype=tf.int32))
        logits = None
        for frame in video:
            frame_in = tf.expand_dims(tf.expand_dims(tf.cast(frame, tf.float32), 0), 0)
            output = streaming_model({**states, "image": frame_in}, training=False)
            logits, states = _unpack_streaming_output(output)
        probs = tf.nn.softmax(logits, axis=-1)[0].numpy()
        pred = int(np.argmax(probs))
        status = "OK " if pred == true_label else "ERR"
        print(f"  [{status}] true={CLASS_NAMES[true_label]:14s} pred={CLASS_NAMES[pred]:14s} conf={probs[pred]:.3f}")
        correct += (pred == true_label); total += 1

print(f"\nStreaming accuracy: {correct}/{total} = {correct/max(1,total):.3f}")


## 💾 Section 9: Export Models

In [ ]:
# (Cell 33 bên dưới đã xử lý export SavedModel + ONNX. Cell này là no-op.)
print("Xem cell 33 để export SavedModel/ONNX.")


In [ ]:
# ==================== Export SavedModel + ONNX (for PC realtime) ====================
SAVED_MODEL_DIR = "movinet_a1_stream_scoop_savedmodel_v1"
ONNX_PATH = "movinet_a1_stream_scoop_v1.onnx"

print("Exporting SavedModel...")
try:
    # tf.saved_model.save() robust hơn model.save(save_format="tf") với custom Model subclass
    tf.saved_model.save(model, SAVED_MODEL_DIR)
    print(f"✓ SavedModel → {SAVED_MODEL_DIR}")
except Exception as e:
    print(f"⚠ SavedModel export failed: {e}. Fallback: save weights only.")
    model.save_weights(f"{SAVED_MODEL_DIR}.weights.h5")

if CONFIG["export_onnx"]:
    try:
        import tf2onnx
        print("\nConverting to ONNX...")
        # Streaming model input: single frame (1,1,H,W,3) + state tensors
        # Với MovinetClassifier output_states=True, TF sẽ tự handle state trong SavedModel
        spec = (tf.TensorSpec((None, NUM_FRAMES, RESOLUTION, RESOLUTION, 3), tf.float32, name="video"),)
        onnx_model, _ = tf2onnx.convert.from_keras(
            model, input_signature=spec, opset=CONFIG["onnx_opset"],
            output_path=ONNX_PATH,
        )
        print(f"✓ ONNX → {ONNX_PATH} ({os.path.getsize(ONNX_PATH) / 1e6:.2f} MB)")
    except ImportError:
        print("⚠ tf2onnx chưa cài. pip install tf2onnx==1.16.1")
    except Exception as e:
        print(f"⚠ ONNX conversion error: {e}")


In [ ]:
# ==================== ONNX vs TF parity check ====================
if os.path.exists(ONNX_PATH):
    try:
        import onnxruntime as ort
        sess = ort.InferenceSession(ONNX_PATH, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
        input_name = sess.get_inputs()[0].name
        print(f"ONNX providers: {sess.get_providers()}")

        n_agree = 0; max_diff = 0.0
        for xb, yb in val_ds.take(3):
            tf_out = model(xb, training=False)
            if isinstance(tf_out, dict):
                tf_out = tf_out.get("logits", list(tf_out.values())[0])
            tf_logits = tf_out.numpy()
            onnx_out = sess.run(None, {input_name: xb.numpy().astype(np.float32)})[0]
            n_agree += int((tf_logits.argmax(-1) == onnx_out.argmax(-1)).sum())
            max_diff = max(max_diff, float(np.abs(tf_logits - onnx_out).max()))
        print(f"✓ TF-vs-ONNX argmax agreement on 3 batches. Max logit diff = {max_diff:.4e}")
    except ImportError:
        print("⚠ onnxruntime chưa cài. pip install onnxruntime-gpu")
    except Exception as e:
        print(f"⚠ Parity check error: {e}")
else:
    print("ONNX file không tồn tại — bỏ qua parity check.")


## 📋 Section 10: Final Summary

In [ ]:
# ==================== Final summary V5 ====================
print("=" * 80)
print("TRAINING SUMMARY — MoViNet-A1 Stream V5 (aligned với official recipe)")
print("=" * 80)

print("\nModel configuration:")
print(f"  Architecture : MoViNet-{MODEL_ID.upper()} Stream (Causal, conv_type={CONFIG['conv_type']})")
print(f"  Input        : {NUM_FRAMES} frames × stride {CONFIG['temporal_stride']} × {RESOLUTION}×{RESOLUTION}×3")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  Classes      : {NUM_CLASSES} — {CLASS_NAMES}")
print(f"  Mixed prec.  : {mixed_precision.global_policy().name}")
print(f"  Dropout      : {CONFIG['dropout_rate']}")
print(f"  Stoch depth  : {CONFIG['stochastic_depth_drop_rate']}")

print("\nTraining recipe V5 — align với official movinet_a1_stream_k600_8x8.yaml:")
print(f"  ✓ num_frames=50 stride=5 (match pretrain 10s)")
print(f"  ✓ stochastic_depth_drop_rate=0.2 (official regularizer)")
print(f"  ✓ CCE + label_smoothing 0.1 (bỏ focal loss — over-engineer)")
print(f"  ✓ AdamW weight_decay=3e-5 (official L2)")
print(f"  ✓ EMA decay 0.99 (Polyak paper)")
print(f"  ✓ Random resized crop (area [0.5, 1.0], aspect [0.75, 1.333])")
print(f"  ✓ 2 stages: head-only {CONFIG['stage1_epochs']}ep + full fine-tune {CONFIG['stage2_epochs']}ep")
print(f"  ✓ TTA 10-crop report (1-crop cho model selection)")
print(f"  ✓ Mixed BF16 — RTX 5080 Blackwell native")
print(f"  ✓ NaN-skip train_step (belt+suspenders)")

print("\nPer-stage checkpoints:")
for stage, path in stage_weights.items():
    if os.path.exists(path):
        print(f"  Stage {stage}: {path} ({os.path.getsize(path)/1e6:.2f} MB)")
if best_stage:
    print(f"  Best overall: Stage {best_stage}, val_acc={best_val_acc:.4f}")

print("\nExport files:")
for p in [SAVED_MODEL_DIR, ONNX_PATH]:
    if os.path.exists(p):
        size = os.path.getsize(p) if os.path.isfile(p) else sum(
            os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(p) for f in fs)
        print(f"  {p}  ({size / 1e6:.2f} MB)")
